# Building a Chatbot
In this notebook consists of design and implementation of an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

In [2]:
import os 
from dotenv import load_dotenv
load_dotenv()
groq_api_key = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "llama-3.3-70b-versatile",groq_api_key = groq_api_key)

/home/udesh_kohli/Code/LLM_Learnings/Langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi, My name is Udesh and I am a Chief AI Engineer")])

AIMessage(content="Nice to meet you, Udesh. It's great to connect with a Chief AI Engineer. That's a fascinating field, and I'm sure you're working on some cutting-edge projects. What kind of AI applications are you currently focusing on, or what are some of the most interesting challenges you're trying to solve?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 49, 'total_tokens': 114, 'completion_time': 0.250364008, 'completion_tokens_details': None, 'prompt_time': 0.002476829, 'prompt_tokens_details': None, 'queue_time': 0.047715668, 'total_time': 0.252840837}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ba38bbab80', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0aaa-d122-7b73-8fc2-e836e86858d3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 65, 'total_tokens': 114})

In [5]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content = "Hi, My name is Udesh and I am a Chief AI Engineer"),
        AIMessage(content ="Nice to meet you, Udesh. It's great to connect with a Chief AI Engineer like yourself. That's a fascinating role, and I'm sure you're working on some cutting-edge projects. What kind of AI applications are you currently focusing on, or what industries are you applying your expertise to? I'm here to chat and learn more about your work." ),
        HumanMessage(content="Hey What is my name and what do I do?")
    ]
)

AIMessage(content='Your name is Udesh, and you are a Chief AI Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 143, 'total_tokens': 158, 'completion_time': 0.042879444, 'completion_tokens_details': None, 'prompt_time': 0.011087534, 'prompt_tokens_details': None, 'queue_time': 0.048026626, 'total_time': 0.053966978}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0aaa-db2b-79c0-8db5-b1f535f3bbce-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 15, 'total_tokens': 158})

### Message Histroy 
we can use a Message Histroy class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. 
Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this! 

In [14]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_session_history(session_id:str)->BaseChatMessageHistory:
    #when ever a session id is given, this funtion check whether the session_id is present in the store dict or not, if present get entire chatMessageHistory from store else create session_id and chat message history
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    print(session_id)
    print(store)
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [7]:
config = {"configurable": {"session_id":"chat1"}}

In [9]:
with_message_history.invoke(
    [HumanMessage(content = "what is my name")], config=config
)

AIMessage(content='Your name is Udesh.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 107, 'total_tokens': 114, 'completion_time': 0.017237319, 'completion_tokens_details': None, 'prompt_time': 0.005401678, 'prompt_tokens_details': None, 'queue_time': 0.092044033, 'total_time': 0.022638997}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0aac-9073-7763-afd8-090e7349245e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 107, 'output_tokens': 7, 'total_tokens': 114})

In [15]:
# change the config --> session id
config1 = {"configurable": {"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content = "My name is john")], config = config1
)
response.content

chat2
{'chat2': InMemoryChatMessageHistory(messages=[])}


"Hello John, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [16]:
with_message_history.invoke(
    [HumanMessage(content = "what is my name")], config=config1
)

chat2
{'chat2': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is john', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello John, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 39, 'total_tokens': 65, 'completion_time': 0.06393989, 'completion_tokens_details': None, 'prompt_time': 0.001925098, 'prompt_tokens_details': None, 'queue_time': 0.049131213, 'total_time': 0.065864988}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0aaf-ee92-7862-bd57-2d83e473bdd6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 26, 'total_tokens': 65})])}


AIMessage(content='Your name is John.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 6, 'prompt_tokens': 78, 'total_tokens': 84, 'completion_time': 0.010191563, 'completion_tokens_details': None, 'prompt_time': 0.003968404, 'prompt_tokens_details': None, 'queue_time': 0.048706688, 'total_time': 0.014159967}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d0ab0-0856-74b1-8776-3560910d446c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 6, 'total_tokens': 84})